
# NYC 311 GenAI Call-Deflection Chatbot Prototype

## Purpose

This notebook builds a GenAI-style prototype that can help offload some common NYC 311 calls by:

- interpreting a user's issue
- matching it to common 311 complaint patterns
- using project outputs from the NYC 311 analysis
- querying the NYC Open Data API for recent/historical complaint patterns
- generating a response with an OpenAI API call

## Required Uploads

Before running this notebook in Colab, upload:

- `notebook2_outputs.zip`
- `notebook3_outputs.zip`

Recommended optional CSV files:

- `top_problem_counts.csv`
- `borough_performance.csv`
- `agency_performance.csv`
- `repeat_complaint_candidates.csv`
- `kpi_summary.csv`
- `channel_usage.csv`
- `delay_rate_by_problem.csv`
- `top_tokens_resolution_description.csv`

## Required API Key

You need an OpenAI API key. The notebook will ask for it using `getpass`, so it is not printed to the notebook output.

## Scope Note

This chatbot does not submit real 311 requests and does not check live request status. It is a call-deflection prototype that provides guidance, routing suggestions, and expected patterns based on public 311 data and project outputs.


In [1]:
# 1. Install and Import Libraries

!pip -q install openai pandas pyarrow requests tabulate

import os
import json
import zipfile
from pathlib import Path
from getpass import getpass
from typing import Dict, Optional

import requests
import pandas as pd
import numpy as np

from google.colab import files


## Upload project output files

Upload `notebook2_outputs.zip`, `notebook3_outputs.zip`, and optional CSV outputs when prompted.

In [3]:
# 2. Upload Files

uploaded = files.upload()

DATA_DIR = Path("/content/nyc311_chatbot_data")
EXTRACT_DIR = DATA_DIR / "extracted"
DATA_DIR.mkdir(exist_ok=True)
EXTRACT_DIR.mkdir(exist_ok=True)

for filename in uploaded.keys():
    src = Path("/content") / filename
    dst = DATA_DIR / filename
    if src.exists():
        if dst.exists():
            dst.unlink()
        src.rename(dst)

print("Uploaded files moved to:", DATA_DIR)
for p in sorted(DATA_DIR.iterdir()):
    print(" -", p.name)


Saving agency_performance.csv to agency_performance.csv
Saving borough_performance.csv to borough_performance.csv
Saving channel_usage.csv to channel_usage.csv
Saving delay_rate_by_problem.csv to delay_rate_by_problem.csv
Saving kpi_summary.csv to kpi_summary.csv
Saving repeat_complaint_candidates.csv to repeat_complaint_candidates.csv
Saving top_problem_counts.csv to top_problem_counts.csv
Saving top_tokens_resolution_description.csv to top_tokens_resolution_description.csv
Uploaded files moved to: /content/nyc311_chatbot_data
 - agency_performance.csv
 - borough_performance.csv
 - channel_usage.csv
 - delay_rate_by_problem.csv
 - extracted
 - kpi_summary.csv
 - notebook2_outputs.zip
 - notebook3_outputs.zip
 - repeat_complaint_candidates.csv
 - top_problem_counts.csv
 - top_tokens_resolution_description.csv


In [4]:
# 3. Extract ZIP Files and Locate Files

for zip_path in DATA_DIR.glob("*.zip"):
    out_dir = EXTRACT_DIR / zip_path.stem
    out_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)
    print("Extracted:", zip_path.name, "->", out_dir)

def find_file(filename: str) -> Optional[Path]:
    search_roots = [DATA_DIR, EXTRACT_DIR]
    for root in search_roots:
        direct = root / filename
        if direct.exists():
            return direct
    for root in search_roots:
        matches = list(root.rglob(filename))
        if matches:
            return matches[0]
    return None


Extracted: notebook2_outputs.zip -> /content/nyc311_chatbot_data/extracted/notebook2_outputs
Extracted: notebook3_outputs.zip -> /content/nyc311_chatbot_data/extracted/notebook3_outputs


## Configure OpenAI API

Paste your OpenAI API key when prompted. It will be stored only in the current Colab runtime.

In [5]:
# 4. Configure OpenAI API

from openai import OpenAI

api_key = getpass("Paste your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# You can change this model if needed.
OPENAI_MODEL = "gpt-4.1-mini"

print("OpenAI client configured.")


Paste your OpenAI API key: ··········
OpenAI client configured.


## Load local project outputs

This step loads the project tables and the prepared sample parquet from Notebook 2.

In [6]:
# 5. Load Local Project Outputs

OPTIONAL_CSVS = [
    "top_problem_counts.csv",
    "borough_performance.csv",
    "agency_performance.csv",
    "repeat_complaint_candidates.csv",
    "kpi_summary.csv",
    "channel_usage.csv",
    "delay_rate_by_problem.csv",
    "top_tokens_resolution_description.csv",
    "top_findings_table.csv",
    "model_metrics.csv",
    "regression_model_metrics.csv",
    "classification_model_metrics.csv",
    "full_counts_complaint_type.csv",
    "full_counts_borough.csv",
    "full_counts_agency.csv",
    "full_monthly_counts.csv",
    "sample_response_time_summary.csv",
    "sample_response_by_complaint_type.csv",
]

tables = {}

for fname in OPTIONAL_CSVS:
    path = find_file(fname)
    if path:
        try:
            tables[fname] = pd.read_csv(path)
            print("Loaded:", fname, tables[fname].shape)
        except Exception as e:
            print("Could not load", fname, ":", e)

sample_path = find_file("analysis_ready_sample.parquet")
if sample_path:
    df_sample = pd.read_parquet(sample_path)
    print("Loaded sample parquet:", df_sample.shape)
else:
    df_sample = pd.DataFrame()
    print("No analysis_ready_sample.parquet found.")


Loaded: top_problem_counts.csv (192, 2)
Loaded: borough_performance.csv (6, 4)
Loaded: agency_performance.csv (15, 4)
Loaded: repeat_complaint_candidates.csv (170, 2)
Loaded: kpi_summary.csv (8, 2)
Loaded: channel_usage.csv (5, 2)
Loaded: delay_rate_by_problem.csv (192, 3)
Loaded: top_tokens_resolution_description.csv (30, 2)
Loaded: top_findings_table.csv (5, 5)
Loaded: model_metrics.csv (6, 11)
Loaded: regression_model_metrics.csv (3, 7)
Loaded: classification_model_metrics.csv (3, 8)
Loaded: full_counts_complaint_type.csv (25, 2)
Loaded: full_counts_borough.csv (7, 2)
Loaded: full_counts_agency.csv (20, 2)
Loaded: full_monthly_counts.csv (76, 2)
Loaded: sample_response_time_summary.csv (10, 2)
Loaded: sample_response_by_complaint_type.csv (25, 4)
Loaded sample parquet: (150000, 39)


## NYC Open Data API helper

This uses the NYC Open Data Socrata API for the 311 Service Requests dataset.

In [7]:
# 6. NYC Open Data API Helpers

NYC_311_API = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

def nyc_api_get(params: Dict, timeout: int = 30) -> pd.DataFrame:
    try:
        r = requests.get(NYC_311_API, params=params, timeout=timeout)
        r.raise_for_status()
        data = r.json()
        return pd.DataFrame(data)
    except Exception as e:
        print("NYC API query failed:", e)
        return pd.DataFrame()

def get_live_top_complaint_types(limit: int = 10) -> pd.DataFrame:
    params = {
        "$select": "complaint_type, count(*) as count",
        "$group": "complaint_type",
        "$order": "count DESC",
        "$limit": limit
    }
    return nyc_api_get(params)

def search_live_311_examples(keyword: str, limit: int = 5) -> pd.DataFrame:
    keyword = keyword.replace("'", "''")
    params = {
        "$select": "created_date, complaint_type, descriptor, agency, borough, status, resolution_description",
        "$where": f"upper(complaint_type) like upper('%{keyword}%') OR upper(descriptor) like upper('%{keyword}%')",
        "$order": "created_date DESC",
        "$limit": limit
    }
    return nyc_api_get(params)

def get_live_agency_for_complaint(complaint_keyword: str, limit: int = 10) -> pd.DataFrame:
    complaint_keyword = complaint_keyword.replace("'", "''")
    params = {
        "$select": "complaint_type, agency, count(*) as count",
        "$where": f"upper(complaint_type) like upper('%{complaint_keyword}%')",
        "$group": "complaint_type, agency",
        "$order": "count DESC",
        "$limit": limit
    }
    return nyc_api_get(params)


## Retrieval helpers

These functions collect evidence from project outputs, sample data, and live NYC Open Data before sending it to the GenAI model.

In [8]:
# 7. Retrieve Context from Local Outputs and Live API

def table_preview(name: str, n: int = 5) -> str:
    if name not in tables or tables[name].empty:
        return ""
    return f"Table: {name}\n" + tables[name].head(n).to_markdown(index=False)

def detect_basic_topic(question: str) -> str:
    q = question.lower()
    if any(x in q for x in ["heat", "hot water", "landlord", "apartment"]):
        return "Heat"
    if any(x in q for x in ["noise", "loud", "music", "neighbor"]):
        return "Noise"
    if any(x in q for x in ["parking", "blocked driveway", "illegal parking"]):
        return "Parking"
    if any(x in q for x in ["trash", "garbage", "sanitation", "missed collection"]):
        return "Sanitation"
    if any(x in q for x in ["tree", "street light", "pothole", "sidewalk"]):
        return "Street"
    return ""

def retrieve_project_context(question: str) -> str:
    q = question.lower()
    parts = []

    if "top_findings_table.csv" in tables:
        parts.append(table_preview("top_findings_table.csv", 5))

    if any(x in q for x in ["top", "complaint", "volume", "common"]):
        for name in ["full_counts_complaint_type.csv", "top_problem_counts.csv"]:
            if name in tables:
                parts.append(table_preview(name, 10))
                break

    if any(x in q for x in ["borough", "brooklyn", "queens", "bronx", "manhattan", "staten"]):
        for name in ["full_counts_borough.csv", "borough_performance.csv"]:
            if name in tables:
                parts.append(table_preview(name, 10))
                break

    if any(x in q for x in ["agency", "department", "handled"]):
        for name in ["full_counts_agency.csv", "agency_performance.csv"]:
            if name in tables:
                parts.append(table_preview(name, 10))
                break

    if any(x in q for x in ["delay", "response", "long", "slow", "time"]):
        for name in ["sample_response_time_summary.csv", "sample_response_by_complaint_type.csv", "delay_rate_by_problem.csv"]:
            if name in tables:
                parts.append(table_preview(name, 10))
                break

    if any(x in q for x in ["model", "predict", "classification", "regression"]):
        for name in ["model_metrics.csv", "regression_model_metrics.csv", "classification_model_metrics.csv"]:
            if name in tables:
                parts.append(table_preview(name, 10))

    if any(x in q for x in ["repeat", "recurring"]):
        if "repeat_complaint_candidates.csv" in tables:
            parts.append(table_preview("repeat_complaint_candidates.csv", 10))

    topic = detect_basic_topic(question)
    if topic and not df_sample.empty:
        text_cols = [c for c in ["complaint_type", "descriptor", "agency", "borough"] if c in df_sample.columns]
        if text_cols:
            mask = pd.Series(False, index=df_sample.index)
            for c in text_cols:
                mask = mask | df_sample[c].astype(str).str.contains(topic, case=False, na=False)
            keep_cols = text_cols + [c for c in ["response_time_hours", "closed_same_day_flag"] if c in df_sample.columns]
            matches = df_sample.loc[mask, keep_cols].head(10)
            if not matches.empty:
                parts.append("Sample matching rows:\n" + matches.to_markdown(index=False))

    return "\n\n".join([p for p in parts if p])

def retrieve_live_context(question: str) -> str:
    q = question.lower()
    parts = []

    topic = detect_basic_topic(question)

    if any(x in q for x in ["top", "common", "highest volume", "most common"]):
        df = get_live_top_complaint_types(10)
        if not df.empty:
            parts.append("Live NYC Open Data top complaint types:\n" + df.to_markdown(index=False))

    if topic:
        examples = search_live_311_examples(topic, 5)
        if not examples.empty:
            parts.append(f"Recent NYC Open Data examples matching '{topic}':\n" + examples.to_markdown(index=False))

        agency = get_live_agency_for_complaint(topic, 10)
        if not agency.empty:
            parts.append(f"Common agencies for '{topic}' complaints:\n" + agency.to_markdown(index=False))

    return "\n\n".join(parts)


## GenAI response generator

This sends the user's question plus retrieved evidence to OpenAI and asks for a practical 311-style response.

In [9]:
# 8. GenAI Response Layer

SYSTEM_INSTRUCTIONS = '''
You are a NYC 311 call-deflection assistant prototype for a data science capstone project.

Your role:
- Interpret the user's issue in plain English.
- Suggest the likely 311 complaint category when evidence supports it.
- Mention the likely responsible agency when evidence supports it.
- Use historical/project evidence to explain expected patterns.
- Give practical next steps.
- Be clear that this prototype does not submit official 311 requests or check live request status.
- For emergencies, safety threats, crimes in progress, fire, or medical issues, tell the user to call 911.

Rules:
- Do not invent exact response times unless provided in the evidence.
- If evidence is limited, say so.
- Keep responses concise but helpful.
- Use a calm, public-service tone.
'''

def genai_answer(question: str, use_live_api: bool = True) -> str:
    project_context = retrieve_project_context(question)
    live_context = retrieve_live_context(question) if use_live_api else ""

    evidence = f'''
PROJECT OUTPUT CONTEXT:
{project_context if project_context else "No project context matched."}

LIVE NYC OPEN DATA CONTEXT:
{live_context if live_context else "No live NYC Open Data context retrieved."}
'''

    user_prompt = f'''
User question:
{question}

Evidence:
{evidence}

Please answer as a 311 call-deflection assistant prototype.
'''

    response = client.responses.create(
        model=OPENAI_MODEL,
        input=[
            {"role": "system", "content": SYSTEM_INSTRUCTIONS},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
    )

    return response.output_text



## Test the chatbot

Examples:

- "My apartment has no heat. What should I do?"
- "There is loud music next door. Is that a 311 issue?"
- "Who handles illegal parking?"
- "What are the most common 311 complaints?"
- "What should leadership focus on based on the analysis?"


In [10]:
# 9. Single Question Test

question = input("Ask the NYC 311 chatbot a question: ")
answer = genai_answer(question, use_live_api=True)
# answer = genai_answer(question, use_live_api=False)
print("\n--- Chatbot Answer ---\n")
print(answer)


Ask the NYC 311 chatbot a question: My apartment has no heat. What should I do?

--- Chatbot Answer ---

You have no heat in your apartment. This is typically reported as a "HEAT/HOT WATER" complaint to the NYC Department of Housing Preservation and Development (HPD).

Based on past data, HPD handles these heat complaints, often inspecting the entire building or individual apartments. Response times vary, but some cases are resolved the same day, while others take longer.

Next steps:
- Contact your building management or landlord immediately to report the issue.
- If they do not resolve it promptly, you can file a heat complaint with HPD through their website or by calling 311.
- Keep records of your communications and any temperatures below required levels (usually below 68°F during heating season).
- For urgent situations, such as no heat during very cold weather, HPD prioritizes inspections.

This prototype cannot submit official complaints or check live status. If you or anyone in

In [11]:
# 10. Multi-turn Chat Loop

while True:
    question = input("\nAsk a 311 question (or type quit): ")
    if question.lower().strip() in ["quit", "exit", "stop"]:
        print("Chat ended.")
        break

    try:
        answer = genai_answer(question, use_live_api=True)
        print("\nAssistant:")
        print(answer)
    except Exception as e:
        print("Error:", e)
        print("If this is an API key or quota issue, check your OpenAI account/API key.")



Ask a 311 question (or type quit): Who handles illegal parking?

Assistant:
Illegal parking complaints in NYC are primarily handled by the NYPD (New York Police Department). This includes issues like blocking sidewalks, hydrants, or violating posted parking signs. Illegal parking is the most common complaint type, with over 2.6 million records historically, indicating it is a frequent concern citywide.

If you encounter illegal parking, you can report it to 311 for NYPD to address. Response times vary, but many cases are resolved the same day. For urgent safety hazards—such as a vehicle blocking emergency access—call 911 immediately.

This prototype does not submit official 311 requests or track live statuses. For official reporting, use NYC 311 channels directly.

Ask a 311 question (or type quit): What should leadership focus on based on the analysis?

Assistant:
Based on the analysis, leadership should focus on addressing the high volume of Illegal Parking complaints, as these repr

## Optional: Save a demo transcript

This saves sample chatbot interactions for your project materials.

In [12]:
# 11. Save Demo Transcript

demo_questions = [


    # Noise complaints
    "What can I do about loud music from my neighbor?",
    "There is construction noise late at night. Can I report that?",
    "A car alarm has been going off for hours. Is that a 311 issue?",
    "My upstairs neighbor is making noise every night. Who handles that?",
    "There is loud street noise from a party outside. What should I do?",

    # Heat and housing
    "My apartment has no heat. What should I do?",
    "My landlord has not fixed hot water for three days.",
    "There is mold in my apartment. Is that a 311 complaint?",
    "My ceiling is leaking. Who do I contact?",
    "The building has broken stairs and no lights in the hallway.",

    # Trash and sanitation
    "Trash was not picked up on my block. Can I report it?",
    "There is illegal dumping on the sidewalk.",
    "Someone left furniture outside for days.",
    "My recycling was missed. What should I do?",
    "There are overflowing garbage cans near my building.",

    # Street and sidewalk issues
    "There is a pothole on my street.",
    "The sidewalk is cracked and dangerous.",
    "A streetlight is out.",
    "The traffic signal is not working.",
    "There is a damaged street sign.",

    # Parking and vehicles
    "A car is blocking my driveway.",
    "There is an abandoned vehicle on my block.",
    "Someone parked in a bus stop.",
    "A vehicle has no plates and has been sitting for weeks.",
    "Can I report illegal parking through 311?",

    # Rodents and pests
    "There are rats in the alley.",
    "I saw rats near a restaurant.",
    "My building has a roach problem.",
    "There are rodents around trash outside.",
    "Who handles pest complaints?",

    # Parks and public spaces
    "There is broken playground equipment.",
    "A park bathroom is dirty.",
    "There is graffiti in the park.",
    "A tree branch fell in the street.",
    "There is a dead tree near my building.",

    # Status and follow up
    "I already filed a complaint. How do I check the status?",
    "My 311 complaint was closed but nothing was fixed.",
    "Why does my complaint say no action taken?",
    "Can I reopen a closed 311 request?",
    "How long does it take to resolve a pothole complaint?",

    # Emergency or escalation
    "There is a fire in my building.",
    "Someone is breaking into a car.",
    "There is a gas smell in my apartment.",
    "A traffic light is out and cars are almost crashing.",
    "A tree fell on a car and someone may be hurt.",

    # Unclear / general questions
    "My block is a mess. What should I report?",
    "Something smells bad outside.",
    "There is a problem with my building.",
    "I need help with a city issue.",
    "I do not know what category my complaint fits into.",

    # Analysis / capstone framing
    "What are the most common 311 complaints?",
    "Which complaint types take the longest to resolve?",
    "Which borough has the most complaints?",
    "What patterns do you see in service request delays?",
    "What should leadership focus on based on the analysis?"
]

transcript = []

for q in demo_questions:
    try:
        # a = genai_answer(q, use_live_api=True)
        a = genai_answer(q, use_live_api=False)
        transcript.append({"question": q, "answer": a})
    except Exception as e:
        transcript.append({"question": q, "answer": f"ERROR: {e}"})

transcript_df = pd.DataFrame(transcript)
transcript_df.to_csv("/content/nyc311_genai_chatbot_demo_transcript.csv", index=False)

display(transcript_df)
print("Saved: /content/nyc311_genai_chatbot_demo_transcript.csv")

,question,answer
0,What can I do about loud music from my neighbor?,You are experiencing loud music from your neig...
1,There is construction noise late at night. Can...,You can report construction noise late at nigh...
2,A car alarm has been going off for hours. Is t...,A car alarm going off for hours is typically c...
3,My upstairs neighbor is making noise every nig...,Your issue is about noise disturbances coming ...
4,There is loud street noise from a party outsid...,You are experiencing loud street noise from a ...
5,My apartment has no heat. What should I do?,You have no heat in your apartment. This issue...
6,My landlord has not fixed hot water for three ...,You are experiencing a lack of hot water in yo...
7,There is mold in my apartment. Is that a 311 c...,"You have mold in your apartment, which is a he..."
8,My ceiling is leaking. Who do I contact?,A leaking ceiling in your home is typically a ...
9,The building has broken stairs and no lights i...,You mentioned broken stairs and no lights in t...


Saved: /content/nyc311_genai_chatbot_demo_transcript.csv



# Notebook Summary

This notebook creates a GenAI-style NYC 311 call-deflection chatbot prototype.

It combines:

- OpenAI API for natural-language understanding and response generation
- NYC Open Data API for live/historical 311 pattern lookup
- project outputs from Notebook 2 and Notebook 3
- prepared sample parquet for local dataset context

This prototype can support common informational 311 questions, but it does not submit official service requests or check official service request status.
